# 18 - Word2Vec + Manual RNN Next Word Prediction

Goal: usar embeddings Word2Vec preentrenados como entrada de una RNN construida desde cero, conectando manualmente el output con el input en cada paso temporal.

Run with: conda activate tfenv

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset
from sklearn.model_selection import train_test_split

print(f"TensorFlow version: {tf.__version__}")

In [ ]:
# Cargar embeddings Word2Vec preentrenados
class Word2VecLoader:
    def __init__(self, path='myWord2Vec/'):
        target_embeddings = np.load(path + 'target_embeddings.npy')
        context_embeddings = np.load(path + 'context_embeddings.npy')
        text_vocab = np.load(path + 'text_vocab.npy', allow_pickle=True).item()

        self.target_embeddings = target_embeddings
        self.context_embeddings = context_embeddings
        self.final_embeddings = (target_embeddings + context_embeddings) / 2
        self.text_vocab = text_vocab
        self.idx_to_word = {idx: word for word, idx in text_vocab.items()}
        self.embedding_dim = target_embeddings.shape[1]

        self.embedding_layer = layers.Embedding(
            input_dim=target_embeddings.shape[0],
            output_dim=target_embeddings.shape[1],
            weights=[target_embeddings],
            trainable=False,
            name='pretrained_embedding'
        )

        print('Embeddings cargados:', target_embeddings.shape, context_embeddings.shape)
        print('Vocabulario cargado:', len(text_vocab))

    def encode(self, words):
        return [self.text_vocab[w] for w in words if w in self.text_vocab]

    def decode(self, token_id):
        return self.idx_to_word.get(int(token_id), '<unk>')

loader = Word2VecLoader()

In [ ]:
# Preparar datos de texto y crear secuencias contexto -> siguiente palabra
print('Loading gaianet/london dataset...')
ds = load_dataset('gaianet/london', split='train')
texts = [row['text'] if 'text' in row else row.get('content', '') for row in ds][:10000]
full_text = ' '.join(texts[:5000])

words = full_text.lower().split()
words = [w.strip('.,;:!?()[]"-0123456789') for w in words]
words = [w for w in words if len(w) > 2 and w in loader.text_vocab]
print(f'Total words used: {len(words)}')
print(f'Sample: {words[:20]}')

def create_sequences(words, window=3):
    X, y = [], []
    for i in range(len(words) - window):
        context = words[i:i + window]
        target = words[i + window]
        if all(w in loader.text_vocab for w in context + [target]):
            X.append([loader.text_vocab[w] for w in context])
            y.append(loader.text_vocab[target])
    return np.array(X, dtype=np.int32), np.array(y, dtype=np.int32)

X, y = create_sequences(words, window=3)
print(f'Sequences created: {len(X)}')
print(f'Input shape: {X.shape}, Target shape: {y.shape}')

if len(X) == 0:
    raise ValueError('No se pudieron crear secuencias con el vocabulario cargado.')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=True)

In [ ]:
# RNN manual desde cero: h_t = tanh(Wx x_t + Wh h_{t-1} + b)
class ManualRNNWordPredictor(keras.Model):
    def __init__(self, embedding_layer, vocab_size, embed_dim, hidden_dim=64):
        super().__init__()
        self.embedding = embedding_layer
        self.hidden_dim = hidden_dim
        self.Wx = layers.Dense(hidden_dim, use_bias=False)
        self.Wh = layers.Dense(hidden_dim, use_bias=False)
        self.b = self.add_weight(shape=(hidden_dim,), initializer='zeros', trainable=True, name='bias')
        self.out = layers.Dense(vocab_size, activation='softmax')

    def call(self, inputs):
        x = self.embedding(inputs)
        h = tf.zeros((tf.shape(x)[0], self.hidden_dim))
        # Conexión manual del output con el input mediante el estado oculto h
        for xt in tf.unstack(x, axis=1):
            h = tf.tanh(self.Wx(xt) + self.Wh(h) + self.b)
        return self.out(h)

vocab_size = loader.target_embeddings.shape[0]
model = ManualRNNWordPredictor(loader.embedding_layer, vocab_size, loader.embedding_dim, hidden_dim=64)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.build((None, 3))
model.summary()

batch_size = 64
train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train)).shuffle(min(len(X_train), 2000)).batch(batch_size).prefetch(tf.data.AUTOTUNE)
val_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(batch_size).prefetch(tf.data.AUTOTUNE)

history = model.fit(train_ds, validation_data=val_ds, epochs=10, verbose=1)

In [ ]:
# Evaluación y predicción de siguiente palabra
from sklearn.metrics import accuracy_score

loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f'Test accuracy: {acc:.3f}')

def predict_next_word(context_words, top_n=5):
    context_ids = [loader.text_vocab[w] for w in context_words if w in loader.text_vocab]
    if len(context_ids) < 3:
        return []
    context_ids = np.array([context_ids[:3]], dtype=np.int32)
    probs = model.predict(context_ids, verbose=0)[0]
    top_indices = np.argsort(probs)[-top_n:][::-1]
    return [(loader.decode(idx), float(probs[idx])) for idx in top_indices]

sample_contexts = [
    ['london', 'bridge', 'river'],
    ['bank', 'of', 'england'],
    ['queen', 'of', 'england']
]

for context in sample_contexts:
    preds = predict_next_word(context, top_n=5)
    if preds:
        print(f"Contexto {context} -> {preds}")

plt.figure(figsize=(8, 4))
plt.plot(history.history['accuracy'], label='train_acc')
plt.plot(history.history['val_accuracy'], label='val_acc')
plt.title('Manual RNN training history')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()